# Training Demo

This notebook demonstrates the full training pipeline on a small synthetic dataset.
It trains EDSR for a few steps to verify everything works end-to-end.

For real training, use the CLI:
```bash
python train.py --config configs/edsr_x4.yaml
```

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from omegaconf import OmegaConf
import matplotlib.pyplot as plt

import models
from models.cnn.edsr import EDSR
from losses import PixelLoss
from metrics import compute_psnr, compute_ssim

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## Synthetic Dataset
We create a small synthetic dataset using random images to verify the pipeline.

In [ ]:
SCALE = 4
N = 64       # number of synthetic images
LR_SIZE = 24 # LR patch size
HR_SIZE = LR_SIZE * SCALE

torch.manual_seed(42)
hr_data = torch.rand(N, 3, HR_SIZE, HR_SIZE)
lr_data = F.interpolate(hr_data, scale_factor=1/SCALE, mode='bicubic', align_corners=False).clamp(0, 1)

dataset = TensorDataset(lr_data, hr_data)
loader = DataLoader(dataset, batch_size=8, shuffle=True)
print(f'Synthetic dataset: {len(dataset)} pairs | LR {LR_SIZE}² → HR {HR_SIZE}²')

## Build Model + Optimizer + Loss

In [ ]:
model = EDSR(scale=SCALE, num_features=32, num_resblocks=4).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
loss_fn = PixelLoss(loss_type='L1')
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

print(f'Model params: {model.param_count():,}')

## Training Loop

In [ ]:
NUM_EPOCHS = 20
losses_history = []
psnr_history = []

for epoch in range(NUM_EPOCHS):
    model.train()
    epoch_loss = 0.0
    for lr_batch, hr_batch in loader:
        lr_batch = lr_batch.to(device)
        hr_batch = hr_batch.to(device)
        
        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
            sr = model(lr_batch)
            loss = loss_fn(sr, hr_batch)
        
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        epoch_loss += loss.item()
    
    epoch_loss /= len(loader)
    losses_history.append(epoch_loss)
    
    # Validation PSNR
    model.eval()
    with torch.no_grad():
        sr_val = model(lr_data[:8].to(device))
        psnr_val = compute_psnr(sr_val.cpu(), hr_data[:8])
    psnr_history.append(psnr_val)
    
    if (epoch + 1) % 5 == 0:
        print(f'Epoch [{epoch+1:3d}/{NUM_EPOCHS}]  Loss: {epoch_loss:.4f}  PSNR: {psnr_val:.2f} dB')

## Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(losses_history)
ax1.set_title('L1 Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.grid(True)

ax2.plot(psnr_history)
ax2.set_title('PSNR (dB)')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('PSNR')
ax2.grid(True)

plt.tight_layout()
plt.show()

## Visual Comparison

In [ ]:
model.eval()
idx = 0
with torch.no_grad():
    lr_img = lr_data[idx:idx+1].to(device)
    sr_img = model(lr_img).cpu()
    bicubic_img = F.interpolate(lr_img.cpu(), scale_factor=SCALE, mode='bicubic', align_corners=False).clamp(0, 1)
    hr_img = hr_data[idx]

def to_np(t):
    return t.squeeze(0).permute(1, 2, 0).numpy().clip(0, 1)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
titles = ['LR Input', 'Bicubic ×4', 'EDSR SR', 'HR Ground Truth']
images = [to_np(lr_data[idx].unsqueeze(0)), to_np(bicubic_img), to_np(sr_img), to_np(hr_img.unsqueeze(0))]

for ax, img, title in zip(axes, images, titles):
    ax.imshow(img)
    ax.set_title(title)
    ax.axis('off')

plt.tight_layout()
plt.show()

print(f'PSNR - Bicubic: {compute_psnr(bicubic_img, hr_img.unsqueeze(0)):.2f} dB')
print(f'PSNR - EDSR:    {compute_psnr(sr_img, hr_img.unsqueeze(0)):.2f} dB')

## Config-based Training

For full training with OmegaConf config, use the CLI:

```bash
# Train EDSR ×4
python train.py --config configs/edsr_x4.yaml

# Train SwinIR ×4 with mixed precision
python train.py --config configs/swinir_x4.yaml

# Train ESRGAN ×4 (GAN training with pretraining phase)
python train.py --config configs/esrgan_x4.yaml

# Resume training from checkpoint
python train.py --config configs/edsr_x4.yaml --resume outputs/edsr_x4/epoch_0100.pth
```

Config keys you'll most likely want to change for your GPU:

| Key | Default | RTX 3050 Ti (4GB) |
|-----|---------|-------------------|
| `data.batch_size` | 16 | 4-8 |
| `data.patch_size` | 48 | 32-64 |
| `trainer.grad_accumulation_steps` | 1 | 4-16 |
| `trainer.mixed_precision` | true | true |